# Filled ERD Generator for Reprint Groups

This notebook generates filled Entity-Relationship Diagrams for articles in a specific reprint group.

**What it does:**
- Filters your CSV by a `group_reprint_id` value
- Creates one ERD per article with actual data filled in
- Outputs PNG and SVG files

**Usage:**
1. Run the function definition cells
2. Modify the parameters in the final cell
3. Run it!

## 1. Import Libraries

In [1]:
import pandas as pd
from graphviz import Digraph
from pathlib import Path


## 2. Helper Function - HTML Escaping

In [2]:
def escape_html(text):
    """Escape special characters for HTML in graphviz"""
    if pd.isna(text) or text == '':
        return ''
    text = str(text)
    text = text.replace('&', '&amp;')
    text = text.replace('<', '&lt;')
    text = text.replace('>', '&gt;')
    text = text.replace('"', '&quot;')
    # Truncate very long text
    if len(text) > 100:
        text = text[:100] + '...'
    return text

print("✓ Helper function defined!")

✓ Helper function defined!


## 3. Main Function - Create Filled ERD for Single Article

In [3]:
def create_filled_erd_for_article(article_id, df_all, output_dir='output_erds'):
    """
    Create a filled ERD for a single article with all its actual data.
    Handles multiple images per article by showing count and sample.
    """
    
    # Get all records (images) for this article
    article_records = df_all[df_all['article_id'] == article_id]
    
    if len(article_records) == 0:
        print(f"Warning: No records found for article {article_id}")
        return None
    
    # Get the first record for article-level data
    article_data = article_records.iloc[0]
    
    # Create directed graph
    erd = Digraph(f'ERD_{article_id}', format='png')
    erd.attr(rankdir='LR', splines='polyline', nodesep='1.0', ranksep='1.8')
    erd.attr('node', shape='plaintext')
    erd.attr('edge', fontsize='9', penwidth='1.5')
    
    # Add title
    reprint_type = escape_html(article_data['reprint_type'])
    title = f'Entity Relationships for {escape_html(article_id)}\\n(Reprint Type: {reprint_type})'
    erd.attr(label=title, fontsize='14', labelloc='t', fontname='Arial Bold')
    
    # --- ARTICLE ENTITY ---
    article_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#B4D7E5" COLSPAN="2"><FONT POINT-SIZE="12"><B>Article</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>article_id:</B> {escape_html(article_data['article_id'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>title:</B> {escape_html(article_data['title'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>author:</B> {escape_html(article_data['author'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>date:</B> {escape_html(article_data['date'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>monograph_publishers:</B> {escape_html(article_data['monograph_publishers'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>volume_number:</B> {escape_html(article_data['volume_number'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>issue_number:</B> {escape_html(article_data['issue_number'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>pages:</B> {escape_html(article_data['pages'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>archival_holding:</B> {escape_html(article_data['arhival_holding'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>archival_location:</B> {escape_html(article_data['archival_location'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>cost:</B> {escape_html(article_data['cost'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>permission:</B> {escape_html(article_data['permission'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>permission_acknowledgement:</B> {escape_html(article_data['permission_acknowledgement'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>permission_statements:</B> {escape_html(article_data['permission_statements'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>acquisitions:</B> {escape_html(article_data['acquisitions'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>document_intro:</B> {escape_html(article_data['document_intro'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>transcripts:</B> {escape_html(article_data['transcripts'])}</TD></TR>
  <TR><TD ALIGN="LEFT"><B>FK</B></TD><TD ALIGN="LEFT"><B>publisher_location:</B> {escape_html(article_data['publisher_location'])}</TD></TR>
  <TR><TD ALIGN="LEFT"><B>FK</B></TD><TD ALIGN="LEFT"><B>publication:</B> {escape_html(article_data['publication'])}</TD></TR>
</TABLE>>'''
    
    erd.node('Article', label=article_html)
    
    # --- PUBLICATION ENTITY ---
    pub_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#F4D9B0" COLSPAN="2"><FONT POINT-SIZE="12"><B>Publication</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>publication:</B> {escape_html(article_data['publication'])}</TD></TR>
  <TR><TD ALIGN="LEFT"><B>FK</B></TD><TD ALIGN="LEFT"><B>publisher_location:</B> {escape_html(article_data['publisher_location'])}</TD></TR>
</TABLE>>'''
    
    erd.node('Publication', label=pub_html)
    
    # --- LOCATION ENTITY ---
    loc_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#C8E6C9" COLSPAN="2"><FONT POINT-SIZE="12"><B>Location</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>publisher_location:</B> {escape_html(article_data['publisher_location'])}</TD></TR>
</TABLE>>'''
    
    erd.node('Location', label=loc_html)
    
    # --- IMAGES ENTITY ---
    num_images = len(article_records)
    images_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#E1BEE7" COLSPAN="2"><FONT POINT-SIZE="12"><B>Images ({num_images} total)</B></FONT></TD></TR>'''
    
    # Show up to 3 sample images
    for idx, (_, img_row) in enumerate(article_records.head(3).iterrows()):
        images_html += f'  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>object_id:</B> {escape_html(img_row["object_id"])}</TD></TR>\n'
        images_html += f'  <TR><TD></TD><TD ALIGN="LEFT"><B>image_file:</B> {escape_html(img_row["image_file"])}</TD></TR>\n'
        images_html += f'  <TR><TD></TD><TD ALIGN="LEFT"><B>image_parent_id:</B> {escape_html(img_row["image_parent_id"])}</TD></TR>\n'
        images_html += f'  <TR><TD></TD><TD ALIGN="LEFT"><B>image_object_location:</B> {escape_html(img_row["image_object_location"])}</TD></TR>\n'
        images_html += f'  <TR><TD></TD><TD ALIGN="LEFT"><B>image_display_template:</B> {escape_html(img_row["image_display_template"])}</TD></TR>\n'
        images_html += f'  <TR><TD></TD><TD ALIGN="LEFT"><B>image_description:</B> {escape_html(img_row["image_description"])}</TD></TR>\n'
        
        if idx < 2 and idx < num_images - 1:
            images_html += f'  <TR><TD COLSPAN="2" BGCOLOR="#F0F0F0">---</TD></TR>\n'
    
    if num_images > 3:
        images_html += f'  <TR><TD COLSPAN="2" ALIGN="CENTER"><I>... and {num_images - 3} more images</I></TD></TR>\n'
    
    images_html += f'  <TR><TD ALIGN="LEFT"><B>FK</B></TD><TD ALIGN="LEFT"><B>article_id:</B> {escape_html(article_data["article_id"])}</TD></TR>\n'
    images_html += '</TABLE>>'
    
    erd.node('Images', label=images_html)
    
    # --- REPRINTS ENTITY ---
    # Collect all document groups this article belongs to
    all_groups = df_all[df_all['article_id'] == article_id]['group_reprint_id'].unique().tolist()
    groups_display = ', '.join(escape_html(g) for g in all_groups[:3])
    if len(all_groups) > 3:
        groups_display += f', ... ({len(all_groups)} total)'

    reprints_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#FFECB3" COLSPAN="2"><FONT POINT-SIZE="12"><B>Reprints / Document Groups</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>group_reprint_id:</B> {groups_display}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>reprint_type:</B> {escape_html(article_data['reprint_type'])}</TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>reprint_order:</B> </TD></TR>
</TABLE>>'''
    
    erd.node('Reprints', label=reprints_html)
    
    # --- TROPES ---
    tropes_list = []
    if pd.notna(article_data['tropes']) and article_data['tropes'] != '':
        tropes_list = [t.strip() for t in str(article_data['tropes']).split(';')]
    
    if tropes_list:
        # Article_Trope junction table
        article_trope_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#FFE0E0" COLSPAN="2"><FONT POINT-SIZE="12"><B>article_trope</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK,FK</B></TD><TD ALIGN="LEFT"><B>article_id:</B> {escape_html(article_data['article_id'])}</TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK,FK</B></TD><TD ALIGN="LEFT"><B>trope_id:</B></TD></TR>'''
        
        for trope in tropes_list:
            article_trope_html += f'  <TR><TD COLSPAN="2" ALIGN="CENTER">{escape_html(trope)}</TD></TR>\n'
        
        article_trope_html += '</TABLE>>'
        erd.node('Article_Trope', label=article_trope_html)
        
        # Trope entity
        trope_html = f'''<
<TABLE BORDER="2" CELLBORDER="1" CELLSPACING="0" CELLPADDING="6">
  <TR><TD BGCOLOR="#FFCDD2" COLSPAN="2"><FONT POINT-SIZE="12"><B>trope</B></FONT></TD></TR>
  <TR><TD ALIGN="LEFT"><B>PK</B></TD><TD ALIGN="LEFT"><B>trope_id:</B></TD></TR>
  <TR><TD></TD><TD ALIGN="LEFT"><B>trope_label:</B></TD></TR>'''
        
        for trope in tropes_list:
            trope_html += f'  <TR><TD COLSPAN="2" ALIGN="CENTER">{escape_html(trope)}</TD></TR>\n'
        
        trope_html += '</TABLE>>'
        erd.node('Trope', label=trope_html)
    
    # --- ADD RELATIONSHIPS ---
    erd.edge('Article', 'Publication', xlabel='published in\n(N:1)', 
             arrowhead='none', arrowtail='crow', dir='both')
    erd.edge('Article', 'Location', xlabel='published at\n(N:1)', 
             arrowhead='none', arrowtail='crow', dir='both')
    erd.edge('Publication', 'Location', xlabel='located at\n(N:1)', 
             arrowhead='none', arrowtail='crow', dir='both')
    erd.edge('Article', 'Images', xlabel='has\n(1:N)', 
             arrowhead='crow', arrowtail='none', dir='both')
    # Many-to-many: an article can belong to multiple document groups
    erd.edge('Article', 'Reprints', xlabel='belongs to\n(N:M)', 
             arrowhead='crow', arrowtail='crow', dir='both')
    
    if tropes_list:
        erd.edge('Article', 'Article_Trope', xlabel='(1:N)', 
                 arrowhead='crow', arrowtail='none', dir='both')
        erd.edge('Trope', 'Article_Trope', xlabel='(1:N)', 
                 arrowhead='crow', arrowtail='none', dir='both')
    
    # Create output directory
    Path(output_dir).mkdir(exist_ok=True)
    
    # Render
    output_path = f'{output_dir}/{article_id}_erd'
    try:
        erd.render(output_path, cleanup=True, format='png')
        erd.render(output_path + '_svg', cleanup=True, format='svg')
        print(f"  ✓ Created ERD for {article_id}")
        return output_path
    except Exception as e:
        print(f"  ✗ Error creating ERD for {article_id}: {e}")
        return None

print("✓ Main ERD function defined!")


✓ Main ERD function defined!


## 4. Wrapper Function - Generate ERDs for Entire Reprint Group

In [4]:
def generate_erds_for_reprint_group(csv_file, reprint_group_id, output_dir='output_erds'):
    """
    Generate filled ERDs for all articles in a specific reprint group.
    
    Args:
        csv_file: Path to CSV file
        reprint_group_id: Value to filter on in group_reprint_id column
        output_dir: Directory to save output ERDs
    """
    
    # Read CSV
    print(f"Reading CSV: {csv_file}")
    df = pd.read_csv(csv_file)
    print(f"Total records in CSV: {len(df)}")
    
    # Filter by reprint group
    print(f"\nFiltering for group_reprint_id = '{reprint_group_id}'")
    df_group = df[df['group_reprint_id'] == reprint_group_id]
    
    if len(df_group) == 0:
        print(f"\n❌ No records found for reprint group '{reprint_group_id}'")
        print("\nAvailable reprint groups in this CSV:")
        for group in df['group_reprint_id'].unique():
            count = len(df[df['group_reprint_id'] == group])
            articles = df[df['group_reprint_id'] == group]['article_id'].nunique()
            print(f"  - {group}: {count} records, {articles} unique articles")
        return
    
    print(f"Found {len(df_group)} records in this reprint group")
    
    # Get unique articles
    unique_articles = df_group['article_id'].unique()
    print(f"Unique articles: {len(unique_articles)}")
    
    # Print details
    print(f"\nArticles in '{reprint_group_id}':")
    for article_id in unique_articles:
        article_data = df_group[df_group['article_id'] == article_id].iloc[0]
        num_images = len(df_group[df_group['article_id'] == article_id])
        print(f"  - {article_id}")
        print(f"    Title: {article_data['title']}")
        print(f"    Reprint Type: {article_data['reprint_type']}")
        print(f"    Date: {article_data['date']}")
        print(f"    Images: {num_images}")
    
    # Generate ERDs
    print(f"\n{'='*60}")
    print(f"Generating ERDs for {len(unique_articles)} articles...")
    print(f"{'='*60}\n")
    
    created_erds = []
    for article_id in unique_articles:
        result = create_filled_erd_for_article(article_id, df_group, output_dir)
        if result:
            created_erds.append(result)
    
    # Summary
    print(f"\n{'='*60}")
    print(f"✓ Successfully created {len(created_erds)} ERDs")
    print(f"  Output directory: {output_dir}/")
    print(f"  Files: PNG and SVG formats")
    print(f"{'='*60}")

print("✓ Wrapper function defined!")

✓ Wrapper function defined!


## 5. Run It!

**Modify these parameters and run the cell:**

In [5]:
# ===== MODIFY THESE PARAMETERS =====

csv_file = '/Users/owenmonroe/Desktop/GitHub/lonewoman/complete_metadata_images_tropes_reprints_transcripts.csv'  # Path to your CSV file
reprint_group_id = 'TheBostonAtlas1847_reprint'  # The reprint group you want to visualize
output_dir = '/Users/owenmonroe/Desktop/GitHub/lonewoman/ERD/ERD_output'  # Where to save the ERD files

# ===== RUN THE GENERATOR =====

generate_erds_for_reprint_group(csv_file, reprint_group_id, output_dir)

Reading CSV: /Users/owenmonroe/Desktop/GitHub/lonewoman/complete_metadata_images_tropes_reprints_transcripts.csv
Total records in CSV: 4428

Filtering for group_reprint_id = 'TheBostonAtlas1847_reprint'
Found 120 records in this reprint group
Unique articles: 14

Articles in 'TheBostonAtlas1847_reprint':
  - ChristianAdvocateAndJournal1847
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1847-03-03
    Images: 10
  - EdgefieldAdvertiser1847
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1847-02-10
    Images: 9
  - LongIslander1848
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1848-01-21
    Images: 7
  - NewBedfordMercury1847
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1847-01-15
    Images: 9
  - NewYorkDailyTribune1847
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1847-01-18
    Images: 9
  - NewYorkEveningExpress1847
    Title: "A Female Crusoe"
    Reprint Type: direct
    Date: 1847-01-08
    

## Optional: List All Available Reprint Groups

Run this cell to see what reprint groups are in your CSV:

In [8]:
# Load your CSV
csv_file = 'your_complete_metadata.csv'  # Update this path

df = pd.read_csv('/Users/owenmonroe/Desktop/GitHub/lonewoman/_data/cb_complete_metadata_images_tropes_reprints_transcripts.csv')

print("Available reprint groups:\n")
for group in df['group_reprint_id'].unique():
    count = len(df[df['group_reprint_id'] == group])
    articles = df[df['group_reprint_id'] == group]['article_id'].nunique()
    print(f"  📁 {group}")
    print(f"     - {count} total records")
    print(f"     - {articles} unique articles\n")

Available reprint groups:

  📁 Goliah1853_DPT_reprint
     - 49 total records
     - 7 unique articles

  📁 Stuart1878_reprint
     - 451 total records
     - 61 unique articles

  📁 Hardacre1880_SM_reprint
     - 388 total records
     - 28 unique articles

  📁 FHDudley1901_reprint
     - 132 total records
     - 10 unique articles

  📁 Fife1943_reprint
     - 9 total records
     - 3 unique articles

  📁 nan
     - 0 total records
     - 0 unique articles

  📁 Roberts1886_reprint
     - 65 total records
     - 12 unique articles

  📁 SanFranciscoExaminer1892_reprint
     - 146 total records
     - 24 unique articles

  📁 Witter1885_DAC_reprint
     - 272 total records
     - 34 unique articles

  📁 TheDailyHerald1853_reprint
     - 44 total records
     - 11 unique articles

  📁 Bennett1898_reprint
     - 16 total records
     - 2 unique articles

  📁 Blanco1882_TIS_reprint
     - 27 total records
     - 2 unique articles

  📁 Bowers1892_reprint
     - 25 total records
     - 3 uniqu

## Tips

- **Run cells in order** from top to bottom the first time
- **After that**, you only need to modify and re-run the "Run It!" cell
- **Output files** are saved in the directory you specify
- **Both formats**: PNG (for viewing) and SVG (for editing/web) are created

## Troubleshooting

**If you get "No module named 'graphviz'":**
```bash
pip install graphviz pandas
```

**If you get "graphviz executable not found":**
- **Mac**: `brew install graphviz`
- **Ubuntu/Debian**: `sudo apt-get install graphviz`
- **Windows**: Download from https://graphviz.org/download/
